Mount from adfs to databrick

In [0]:
dbutils.fs.ls("/mnt/raw")

Parameterized notebook  

In [0]:
dbutils.widgets.text("file_name", "")
dbutils.widgets.text("file_date", "")

file_name = dbutils.widgets.get("file_name")
file_date = dbutils.widgets.get("file_date")

print(file_name, file_date)

In [0]:
raw_path = f"/mnt/raw/{file_name}"
bronze_path = "/mnt/bronze"
silver_path = "/mnt/silver"

In [0]:
def log(msg):
    print(f"[LOG] {msg}")

log("Pipeline Started")

File check

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS processed_files (
    file_name STRING,
    file_date STRING,
    processed_time TIMESTAMP
)
""")

In [0]:
check_df = spark.sql(f"""
SELECT * FROM processed_files 
WHERE file_name = '{file_name}' 
AND file_date = '{file_date}'
""")

if check_df.count() > 0:
    dbutils.notebook.exit("SKIPPED - Already Processed")

bronze 

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df = spark.read.option("header", True) \
    .option("inferSchema", True) \
    .csv(raw_path)

In [0]:
df = df.withColumn("ingestion_time", current_timestamp()) \
       .withColumn("file_name", lit(file_name)) \
       .withColumn("file_date", lit(file_date))

In [0]:
df.write.format("delta") \
    .mode("append") \
    .save(bronze_path)

log("Bronze Loaded")

silver


In [0]:
df = spark.read.format("delta").load(bronze_path) \
    .filter(f"file_name = '{file_name}' AND file_date = '{file_date}'")

Schema Drift

In [0]:
from pyspark.sql.functions import col, lit

expected_cols = ["order_id","order_date","customer_id","region",
                 "product","category","sales","quantity",
                 "ingestion_time","file_name","file_date"]

for c in expected_cols:
    if c not in df.columns:
        df = df.withColumn(c, lit(None))

df = df.select(expected_cols)

In [0]:
df = df.dropDuplicates(["order_id"])
df = df.filter(col("sales").isNotNull())
df = df.filter(col("sales") > 0)

In [0]:
from delta.tables import DeltaTable

if not DeltaTable.isDeltaTable(spark, silver_path):
    df.write.format("delta").mode("overwrite").save(silver_path)

else:
    silver_table = DeltaTable.forPath(spark, silver_path)

    silver_table.alias("target").merge(
        df.alias("source"),
        "target.order_id = source.order_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

log("Silver Updated")

Insert into process file

In [0]:
from pyspark.sql import Row
from datetime import datetime

new_file = spark.createDataFrame(
    [Row(file_name=file_name, file_date=file_date, processed_time=datetime.now())]
)

new_file.write.mode("append").saveAsTable("processed_files")

log("Tracking Updated")